# CS781 Phase 5 — Experiment 4: Model1 + Prefix Tuning (FAST + Refit + Ensemble)

**Task:** Legal Statute Identification — 100-label multilabel classification  
**Phase-1 foundation:** Model1 — Encoder/Prefix-LM (~120M), 14 layers, d=768, 12 heads, FFN=3072, sinusoidal positions, pre-LN, GELU  
**Adaptation:** **Prefix Tuning**

This experiment is intentionally optimized for a Kaggle T4.

### Prefix design

The pretrained Model1 backbone is frozen. We learn continuous **key/value prefix vectors** which are prepended to attention memory in the **top 6 Transformer layers**.

- prefix length = **16 virtual tokens**
- adapted layers = **8–13**
- bottom 8 layers execute under `torch.no_grad()`
- only prefix tensors + Phase-5 classifier are trainable
- document processing = **HEAD / statute-EVIDENCE / TAIL**
- 320 tokens per view
- 2 heldout epochs
- one DEV inference pass per epoch

Model1 is especially suitable because its Phase-1 self-attention already supports per-layer `past_kv`; the prefix can therefore be inserted without altering pretrained Q/K/V projection weights.

This follows the key Prefix-Tuning principle: freeze pretrained LM parameters and optimize continuous task-specific vectors that subsequent representations attend to as virtual tokens.

Reference: Li & Liang, *Prefix-Tuning: Optimizing Continuous Prompts for Generation*, ACL-IJCNLP 2021.

### Submission outputs

The full run creates:

1. **Heldout Prefix-Tuning ZIP**
2. **Fresh TRAIN+DEV refit ZIP**
3. **Heldout + refit probability ensemble ZIP**

Refit is enabled by default but can be disabled with `RUN_REFIT=False` for a quick run.


### v3 improvements adopted from the modified notebook

The **core Prefix-Tuning implementation is unchanged**: Model1 stays frozen and only per-layer continuous K/V prefixes in layers 8–13 plus the Phase-5 classifier are trained.

Useful additions adopted here:

- **Real-token K/V initialization:** prefixes are initialized from K/V activations produced by a real statute-evidence training chunk, rather than starting entirely from small Gaussian noise.
- **Prefix LR increased to 2e-3** to help the very small prefix parameter set adapt within only two heldout epochs.
- **Prefix length remains 16**, not 32, to preserve the speed advantage.
- **Refit remains enabled**, because we want heldout, refit, and heldout+refit ensemble submissions.


## Kaggle inputs

Attach:

1. Model1 Phase-1 **`enc_full_best*.pt`**
2. its exact **`tokenizer.json`**

The notebook searches `/kaggle/input` automatically. If needed, use the override variables in Cell 1.


In [3]:
# CELL 1 — imports + exact Model1 / Prefix-Tuning config
import os, re, gc, json, math, time, random, hashlib, zipfile, shutil, contextlib, subprocess, sys, importlib.util
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score
from huggingface_hub import hf_hub_download, login

SEED = 3456
NOTEBOOK_VERSION = "exp4_model1_prefix_fast_v3_realinit_refit_ensemble"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

ROOT = Path("/kaggle/working/phase5_exp4_model1_prefix_fast_v3")
CACHE_DIR = ROOT / "cache"
ROOT.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ---- exact Phase-1 Model1 constants ----
VOCAB_SIZE = 24_576
D_MODEL = 768
N_LAYERS = 14
N_HEADS = 12
FFN_DIM = 3072
DROPOUT = 0.0
MAX_POSITIONS = 2048
HEAD_DIM = D_MODEL // N_HEADS

# ---- Phase-5 document setup ----
N_LABELS = 100
N_CHUNKS = 3
CHUNK_LEN = 320
CHAR_WINDOW = 5000
VIEW_NAMES = ["head", "statute_evidence", "tail"]

# ---- FAST Prefix Tuning ----
# Bottom 8 layers frozen under no_grad, top 6 receive trainable KV prefixes.
PREFIX_START_BLOCK = 8
PREFIX_BLOCKS = list(range(PREFIX_START_BLOCK, N_LAYERS))
PREFIX_LEN = 16  # kept short for speed
PREFIX_INIT_STD = 0.02
PREFIX_DROPOUT = 0.05
PREFIX_INIT_FROM_REAL_TOKENS = True  # warm-start K/V prefixes from one real evidence chunk

# ---- training ----
MICRO_BATCH_SIZE = 4
DEV_BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 4       # effective document batch = 16
EPOCHS = 2
PREFIX_LR = 2.0e-3  # faster prefix adaptation; backbone stays frozen
HEAD_LR = 8.0e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
EMA_DECAY = 0.998
GRAD_CLIP = 1.0
USE_AMP = True
NUM_WORKERS = 2

# Refit + ensemble are requested, but can be disabled for a quick debug run.
RUN_REFIT = True  # keep refit + ensemble enabled
REFIT_ENSEMBLE_WEIGHT = 0.50

# ---- threshold tuning ----
THR_GRID = np.round(np.arange(0.02, 0.8001, 0.01), 3)
SHRINK_TAUS = [25.0, 50.0, 100.0, 200.0]
CV_REPEATS = 2

MODEL1_CKPT_OVERRIDE = None
TOKENIZER_OVERRIDE = None

def sha256_file(path, chunk=1<<20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def norm_text(s):
    return re.sub(r"\s+", " ", str(s).lower()).strip()

def text_hash(s):
    return hashlib.sha256(norm_text(s).encode("utf-8")).hexdigest()

def atomic_torch_save(obj, path):
    path = Path(path)
    tmp = Path(str(path) + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)

print("ROOT:", ROOT)
print("Model1 prefix blocks:", PREFIX_BLOCKS)
print("Prefix length:", PREFIX_LEN)
print("Views:", VIEW_NAMES, "chunk_len:", CHUNK_LEN)


Device: cuda
GPU: Tesla T4
ROOT: /kaggle/working/phase5_exp4_model1_prefix_fast_v3
Model1 prefix blocks: [8, 9, 10, 11, 12, 13]
Prefix length: 16
Views: ['head', 'statute_evidence', 'tail'] chunk_len: 320


In [4]:
# CELL 2 — locate Model1 BEST checkpoint + exact tokenizer
all_files = []
for base in ["/kaggle/input", "/kaggle/working"]:
    if os.path.exists(base):
        for root, dirs, files in os.walk(base):
            for f in files:
                all_files.append(os.path.join(root, f))

ckpt_cands = [
    p for p in all_files
    if p.lower().endswith(".pt")
    and "enc_full_best" in os.path.basename(p).lower()
]
# Fallback in case user uploaded a generically named best.pt.
if not ckpt_cands:
    ckpt_cands = [
        p for p in all_files
        if p.lower().endswith(".pt")
        and os.path.basename(p).lower() == "best.pt"
        and any(w in p.lower() for w in ["model1", "encoder", "enc_full"])
    ]

tok_cands = [
    p for p in all_files
    if os.path.basename(p).lower() == "tokenizer (1).json"
]

def asset_score(p, words):
    s = p.lower()
    return sum(10 for w in words if w in s) - len(p) / 100000.0

if MODEL1_CKPT_OVERRIDE:
    MODEL1_CKPT = str(MODEL1_CKPT_OVERRIDE)
else:
    assert ckpt_cands, (
        "Could not find enc_full_best*.pt. "
        "Attach the Phase-1 Model1 best checkpoint."
    )
    MODEL1_CKPT = sorted(
        ckpt_cands,
        key=lambda p: asset_score(
            p, ["model1", "encoder", "hybrid", "enc_full", "phase1"]
        ),
        reverse=True,
    )[0]

if TOKENIZER_OVERRIDE:
    TOKENIZER_FILE = str(TOKENIZER_OVERRIDE)
else:
    assert tok_cands, "Could not find tokenizer.json."
    ckpt_dir = os.path.dirname(MODEL1_CKPT).lower()
    TOKENIZER_FILE = sorted(
        tok_cands,
        key=lambda p: (
            20 if os.path.dirname(p).lower() == ckpt_dir else 0,
            asset_score(p, ["model1", "encoder", "hybrid", "phase1"]),
        ),
        reverse=True,
    )[0]

print("SELECTED MODEL1:", MODEL1_CKPT)
print("SELECTED TOKENIZER:", TOKENIZER_FILE)
print("Model1 SHA256:", sha256_file(MODEL1_CKPT))
print("Tokenizer SHA256:", sha256_file(TOKENIZER_FILE))


SELECTED MODEL1: /kaggle/input/datasets/aaryant23/model1-ckpts/enc_full_best (9).pt
SELECTED TOKENIZER: /kaggle/input/datasets/aaryant23/model1-ckpts/tokenizer (1).json
Model1 SHA256: 017c57f393a55c96a9a24d0f98fc62775f3f4d0459b896eefe7a30437accacfc
Tokenizer SHA256: 9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d


In [5]:
# CELL 3 — exact Phase-1 Model1 LegalBPE tokenizer
# Cell 3 — Custom byte-level BPE (same assignment design, scaled up)
BPE_SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]

class LegalBPE:
    def __init__(self, vocab_size=5000):
        self.vocab_size = int(vocab_size)
        self.merges = []
        self.itos = list(BPE_SPECIALS)
        self.stoi = {t:i for i,t in enumerate(self.itos)}
        self.symbol_bytes = {}
        for b in range(256):
            s = f"{b:02x}"
            self.symbol_bytes[s] = bytes([b])
            self.stoi[s] = len(self.itos)
            self.itos.append(s)
        self._reset_encode_cache()

    @property
    def pad_token_id(self): return self.stoi["<pad>"]
    @property
    def unk_token_id(self): return self.stoi["<unk>"]
    @property
    def bos_token_id(self): return self.stoi["<s>"]
    @property
    def eos_token_id(self): return self.stoi["</s>"]
    def __len__(self): return len(self.itos)

    @staticmethod
    def word_to_symbols(word):
        return [f"{b:02x}" for b in word.encode("utf-8")]

    def _reset_encode_cache(self):
        self._ranks = {(l,r): i for i,(l,r) in enumerate(self.merges)}
        self._word_cache = {}
        self._cache_nmerges = len(self.merges)

    def fit(self, sentences, min_count=2):
        word_counts = Counter()
        seen_chars = 0
        t0 = time.time()

        for sentence in sentences:
            seen_chars += len(sentence)
            word_counts.update(re.findall(r"\S+|\s+", sentence))
            if seen_chars and seen_chars % 10_000_000 < len(sentence):
                print(f"  BPE corpus read: {seen_chars/1e6:.0f}M chars | unique units={len(word_counts):,}", flush=True)

        if min_count > 1:
            word_counts = Counter({w:c for w,c in word_counts.items() if c >= min_count})

        print(f"BPE training units after min_count={min_count}: {len(word_counts):,}")
        print(f"Counting time: {(time.time()-t0)/60:.1f} min")

        words = list(word_counts)
        freqs = [word_counts[w] for w in words]
        syms = [self.word_to_symbols(w) for w in words]

        target_merges = self.vocab_size - len(self.itos)
        pair_counts = Counter()
        pair_words = {}

        for wi, s in enumerate(syms):
            f = freqs[wi]
            for j in range(len(s)-1):
                p = (s[j], s[j+1])
                pair_counts[p] += f
                pair_words.setdefault(p, set()).add(wi)

        heap = [(-c,p) for p,c in pair_counts.items()]
        heapq.heapify(heap)

        for merge_num in range(target_merges):
            best_pair = None
            best_freq = 0
            while heap:
                negc, p = heap[0]
                if pair_counts.get(p,0) == -negc and -negc > 0:
                    best_pair, best_freq = p, -negc
                    break
                heapq.heappop(heap)

            if best_pair is None or best_freq <= 1:
                print("No useful pairs remain.")
                break

            left, right = best_pair
            new_symbol = left + right
            self.merges.append([left,right])
            self.symbol_bytes[new_symbol] = self.symbol_bytes[left] + self.symbol_bytes[right]
            if new_symbol not in self.stoi:
                self.stoi[new_symbol] = len(self.itos)
                self.itos.append(new_symbol)

            touched = set()
            for wi in list(pair_words.get(best_pair, ())):
                old = syms[wi]
                f = freqs[wi]

                # Remove old local pair contributions.
                for j in range(len(old)-1):
                    p = (old[j], old[j+1])
                    pair_counts[p] -= f
                    touched.add(p)

                merged = []
                j = 0
                while j < len(old):
                    if j+1 < len(old) and old[j] == left and old[j+1] == right:
                        merged.append(new_symbol)
                        j += 2
                    else:
                        merged.append(old[j])
                        j += 1
                syms[wi] = merged

                # Add new local pair contributions.
                for j in range(len(merged)-1):
                    p = (merged[j], merged[j+1])
                    pair_counts[p] += f
                    pair_words.setdefault(p, set()).add(wi)
                    touched.add(p)

            pair_words.pop(best_pair, None)

            for p in touched:
                c = pair_counts.get(p,0)
                if c > 0:
                    heapq.heappush(heap, (-c,p))
                else:
                    pair_counts.pop(p, None)

            if (merge_num+1) % 1000 == 0 or merge_num == 0 or merge_num+1 == target_merges:
                print(
                    f"merge {merge_num+1:,}/{target_merges:,} | "
                    f"freq={best_freq:,} | vocab={len(self.itos):,}",
                    flush=True
                )

        self.vocab_size = len(self.itos)
        self._reset_encode_cache()
        print(f"BPE complete: vocab={len(self):,}")
        return self

    def encode_word(self, word):
        if self._cache_nmerges != len(self.merges):
            self._reset_encode_cache()

        hit = self._word_cache.get(word)
        if hit is not None:
            return list(hit)

        symbols = self.word_to_symbols(word)
        ranks = self._ranks

        while len(symbols) > 1:
            best = None
            best_rank = None
            for j in range(len(symbols)-1):
                r = ranks.get((symbols[j], symbols[j+1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best = (symbols[j], symbols[j+1])
                    best_rank = r
            if best is None:
                break

            left, right = best
            merged = []
            j = 0
            while j < len(symbols):
                if j+1 < len(symbols) and symbols[j] == left and symbols[j+1] == right:
                    merged.append(left+right)
                    j += 2
                else:
                    merged.append(symbols[j])
                    j += 1
            symbols = merged

        # Bound cache memory for 500M-token preprocessing.
        if len(self._word_cache) >= 250_000:
            self._word_cache.clear()
        self._word_cache[word] = tuple(symbols)
        return symbols

    def encode(self, text):
        out = []
        unk = self.unk_token_id
        for unit in re.findall(r"\S+|\s+", text):
            for s in self.encode_word(unit):
                out.append(self.stoi.get(s, unk))
        return out

    def decode(self, ids):
        buf = bytearray()
        for idx in ids:
            idx = int(idx)
            if idx < 0 or idx >= len(self.itos):
                raise ValueError(f"invalid token id {idx}")
            tok = self.itos[idx]
            if tok in BPE_SPECIALS:
                continue
            buf.extend(self.symbol_bytes[tok])
        return bytes(buf).decode("utf-8", errors="replace")

    def save_json(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump(
                {"vocab_size":len(self), "merges":self.merges, "itos":self.itos},
                f, ensure_ascii=False
            )

    @classmethod
    def load_json(cls, path):
        obj = json.load(open(path, encoding="utf-8"))
        tok = cls(obj["vocab_size"])
        tok.merges = [list(x) for x in obj["merges"]]
        tok.itos = list(obj["itos"])
        tok.stoi = {t:i for i,t in enumerate(tok.itos)}
        tok.vocab_size = len(tok.itos)

        tok.symbol_bytes = {}
        for b in range(256):
            s=f"{b:02x}"
            tok.symbol_bytes[s]=bytes([b])
        for l,r in tok.merges:
            tok.symbol_bytes[l+r] = tok.symbol_bytes[l] + tok.symbol_bytes[r]

        tok._reset_encode_cache()
        return tok

print("Tokenizer class ready.")

tokenizer = LegalBPE.load_json(TOKENIZER_FILE)
print("Tokenizer vocab:", len(tokenizer))
print(
    "special ids:",
    tokenizer.pad_token_id,
    tokenizer.unk_token_id,
    tokenizer.bos_token_id,
    tokenizer.eos_token_id,
)
assert len(tokenizer) == VOCAB_SIZE
assert tokenizer.pad_token_id == 0
print("✅ tokenizer sanity PASS")


Tokenizer class ready.
Tokenizer vocab: 24576
special ids: 0 1 2 3
✅ tokenizer sanity PASS


In [6]:
# CELL 4 — authenticate/download official Phase-5 data
HF_TOKEN=None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN=UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("HF_TOKEN secret unavailable:",repr(e)[:120])

if HF_TOKEN:
    login(token=HF_TOKEN,add_to_git_credential=False)

HF_REPO="Exploration-Lab/CS-781-Capstone"
FNS={
    "train":"Phase_5_LSI/Train/train.json",
    "dev":"Phase_5_LSI/Train/dev.json",
    "test":"Phase_5_LSI/Test/test.json",
}
paths={
    k:hf_hub_download(repo_id=HF_REPO,repo_type="dataset",filename=v,token=HF_TOKEN)
    for k,v in FNS.items()
}
DATA_SHA256={k:sha256_file(v) for k,v in paths.items()}

def read_json(path):
    with open(path,encoding="utf-8") as f: return json.load(f)

train_raw=read_json(paths["train"])
dev_raw=read_json(paths["dev"])
test_raw=read_json(paths["test"])

print("rows:",len(train_raw),len(dev_raw),len(test_raw))
assert len(train_raw)==42750 and len(dev_raw)==10181 and len(test_raw)==13019

for name,recs,labeled in [("TRAIN",train_raw,True),("DEV",dev_raw,True),("TEST",test_raw,False)]:
    ids=[str(r["id"]) for r in recs]
    assert len(ids)==len(set(ids)),f"{name} duplicate IDs"
    for r in recs:
        assert isinstance(r.get("text"),str)
        if labeled:
            assert isinstance(r.get("labels"),list)
            r["labels"]=sorted(set(int(x) for x in r["labels"]))
            assert all(0<=x<N_LABELS for x in r["labels"])

assert not any("labels" in r for r in test_raw)
print("✅ schema/ID/label sanity PASS")


Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

rows: 42750 10181 13019
✅ schema/ID/label sanity PASS


In [7]:
# CELL 5 — leakage-aware clean DEV + class weights
train_hashes={text_hash(r["text"]) for r in train_raw}
dev_hashes={text_hash(r["text"]) for r in dev_raw}
test_hashes={text_hash(r["text"]) for r in test_raw}

print("text overlap train-dev:",len(train_hashes&dev_hashes))
print("text overlap train-test:",len(train_hashes&test_hashes))
print("text overlap dev-test:",len(dev_hashes&test_hashes))

CLEAN_IDX=np.asarray(
    [i for i,r in enumerate(dev_raw) if text_hash(r["text"]) not in train_hashes],
    dtype=np.int64
)
clean_dev=[dev_raw[i] for i in CLEAN_IDX]
train_records=train_raw
test_records=test_raw

print("official DEV:",len(dev_raw),"clean DEV:",len(clean_dev),"removed:",len(dev_raw)-len(clean_dev))
assert len(clean_dev)>9000

train_counts=np.zeros(N_LABELS,dtype=np.int64)
for r in train_records: train_counts[r["labels"]]+=1

dev_counts=np.zeros(N_LABELS,dtype=np.int64)
for r in clean_dev: dev_counts[r["labels"]]+=1

assert np.all(train_counts>0)
assert np.all(dev_counts>0)

raw_pw=(len(train_records)-train_counts)/train_counts
POS_WEIGHT=np.clip(np.sqrt(raw_pw),1.0,10.0).astype(np.float32)
print("pos_weight min/median/max:",POS_WEIGHT.min(),np.median(POS_WEIGHT),POS_WEIGHT.max())

official_truth=[{"id":str(r["id"]),"gold":[int(x) for x in r["labels"]]} for r in dev_raw]
clean_truth=[{"id":str(r["id"]),"gold":[int(x) for x in r["labels"]]} for r in clean_dev]
json.dump(official_truth,open(ROOT/"dev_truth_official.json","w"))
json.dump(clean_truth,open(ROOT/"dev_truth_clean.json","w"))


text overlap train-dev: 81
text overlap train-test: 109
text overlap dev-test: 62
official DEV: 10181 clean DEV: 10063 removed: 118
pos_weight min/median/max: 1.6261922 7.871644 10.0


In [8]:
# CELL 6 — fast 3-view statute-aware token cache
CUE_RE=re.compile(
    r"\b(section|sections|sec\.?|article|articles|act|acts|rule|rules|"
    r"provision|provisions|ipc|crpc|constitution)\b",re.I
)

def _char_window(text,center,width=CHAR_WINDOW):
    if len(text)<=width: return text
    st=max(0,min(len(text)-width,int(center-width//2)))
    return text[st:st+width]

def select_views(text):
    n=len(text)
    if n<=CHAR_WINDOW:
        return [text,text,text]

    head=text[:CHAR_WINDOW]
    tail=text[-CHAR_WINDOW:]

    hits=[m.start() for m in CUE_RE.finditer(text)]
    if hits:
        best_i=best_j=0; j=0
        for i in range(len(hits)):
            if j<i: j=i
            while j+1<len(hits) and hits[j+1]-hits[i]<=CHAR_WINDOW:
                j+=1
            if j-i>best_j-best_i:
                best_i,best_j=i,j
        center=(hits[best_i]+hits[best_j])//2
        evid=_char_window(text,center)
    else:
        evid=_char_window(text,n//2)

    return [head,evid,tail]

def encode_view(text,view_idx):
    ids=tokenizer.encode(text)
    if not ids: ids=[tokenizer.unk_token_id]
    if len(ids)>CHUNK_LEN:
        if view_idx==0:
            ids=ids[:CHUNK_LEN]
        elif view_idx==2:
            ids=ids[-CHUNK_LEN:]
        else:
            st=(len(ids)-CHUNK_LEN)//2
            ids=ids[st:st+CHUNK_LEN]
    return ids

def cache_split(name,records):
    ids_path=CACHE_DIR/f"{name}_ids.npy"
    len_path=CACHE_DIR/f"{name}_lens.npy"
    meta_path=CACHE_DIR/f"{name}_meta.json"

    dataset_key="dev" if name.startswith("dev") else name
    fingerprint={
        "dataset_sha256":DATA_SHA256[dataset_key],
        "tokenizer_sha256":sha256_file(TOKENIZER_FILE),
        "n":len(records),
        "n_chunks":N_CHUNKS,
        "chunk_len":CHUNK_LEN,
        "char_window":CHAR_WINDOW,
        "views":VIEW_NAMES,
        "ids_hash":hashlib.sha256("|".join(str(r["id"]) for r in records).encode()).hexdigest(),
    }

    if ids_path.exists() and len_path.exists() and meta_path.exists():
        if json.load(open(meta_path))==fingerprint:
            print("✅ reuse token cache:",name)
            return np.load(ids_path,mmap_mode="r"),np.load(len_path,mmap_mode="r")

    arr=np.full((len(records),N_CHUNKS,CHUNK_LEN),tokenizer.pad_token_id,dtype=np.uint16)
    lens=np.zeros((len(records),N_CHUNKS),dtype=np.uint16)

    t0=time.time()
    for i,r in enumerate(records):
        for c,txt in enumerate(select_views(r["text"])):
            z=encode_view(txt,c)
            L=min(len(z),CHUNK_LEN)
            arr[i,c,:L]=np.asarray(z[:L],dtype=np.uint16)
            lens[i,c]=L
        if (i+1)%2500==0 or i+1==len(records):
            print(f"{name}: {i+1}/{len(records)} elapsed={(time.time()-t0)/60:.1f}m",flush=True)

    np.save(ids_path,arr); np.save(len_path,lens)
    json.dump(fingerprint,open(meta_path,"w"),indent=2)
    del arr,lens; gc.collect()
    return np.load(ids_path,mmap_mode="r"),np.load(len_path,mmap_mode="r")

train_tok,train_lens=cache_split("train",train_records)
official_tok,official_lens=cache_split("dev",dev_raw)
test_tok,test_lens=cache_split("test",test_records)

print("cache shapes:",train_tok.shape,official_tok.shape,test_tok.shape)
assert train_tok.shape==(len(train_records),N_CHUNKS,CHUNK_LEN)


train: 2500/42750 elapsed=0.1m
train: 5000/42750 elapsed=0.2m
train: 7500/42750 elapsed=0.4m
train: 10000/42750 elapsed=0.5m
train: 12500/42750 elapsed=0.6m
train: 15000/42750 elapsed=0.7m
train: 17500/42750 elapsed=0.8m
train: 20000/42750 elapsed=1.0m
train: 22500/42750 elapsed=1.1m
train: 25000/42750 elapsed=1.2m
train: 27500/42750 elapsed=1.3m
train: 30000/42750 elapsed=1.4m
train: 32500/42750 elapsed=1.5m
train: 35000/42750 elapsed=1.6m
train: 37500/42750 elapsed=1.8m
train: 40000/42750 elapsed=1.9m
train: 42500/42750 elapsed=2.0m
train: 42750/42750 elapsed=2.0m
dev: 2500/10181 elapsed=0.1m
dev: 5000/10181 elapsed=0.2m
dev: 7500/10181 elapsed=0.4m
dev: 10000/10181 elapsed=0.5m
dev: 10181/10181 elapsed=0.5m
test: 2500/13019 elapsed=0.1m
test: 5000/13019 elapsed=0.2m
test: 7500/13019 elapsed=0.4m
test: 10000/13019 elapsed=0.5m
test: 12500/13019 elapsed=0.6m
test: 13019/13019 elapsed=0.7m
cache shapes: (42750, 3, 320) (10181, 3, 320) (13019, 3, 320)


In [9]:
# CELL 7 — exact Phase-1 Model1 architecture
# Cell 9 — Optimized ~120M Prefix Encoder LM with SDPA + KV cache
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_positions):
        super().__init__()
        pos = torch.arange(max_positions, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe = torch.zeros(max_positions, d_model, dtype=torch.float32)
        pe[:,0::2] = torch.sin(pos*div)
        pe[:,1::2] = torch.cos(pos*div)
        self.register_buffer("pe", pe, persistent=False)

    def forward(self, position_ids):
        return self.pe[position_ids]


class PrefixSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3*d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = dropout

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False):
        B,T,C = x.shape
        qkv = self.qkv(x).view(B,T,3,self.n_heads,self.head_dim)
        q,k,v = qkv.unbind(dim=2)
        q = q.transpose(1,2)
        k = k.transpose(1,2)
        v = v.transpose(1,2)

        if past_kv is not None:
            pk,pv = past_kv
            k = torch.cat([pk,k], dim=2)
            v = torch.cat([pv,v], dim=2)

        y = F.scaled_dot_product_attention(
            q,k,v,
            attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=False,
        )
        y = y.transpose(1,2).contiguous().view(B,T,C)
        y = self.out_proj(y)

        new_kv = (k,v) if use_cache else None
        return y, new_kv


class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, ffn_dim, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = PrefixSelfAttention(d_model,n_heads,dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model,ffn_dim),
            nn.GELU(),
            nn.Linear(ffn_dim,d_model),
            nn.Dropout(dropout),
        )
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False):
        a,new_kv = self.attn(
            self.ln1(x),
            attn_mask=attn_mask,
            past_kv=past_kv,
            use_cache=use_cache,
        )
        x = x + self.resid_drop(a)
        x = x + self.ff(self.ln2(x))
        return x,new_kv


class PrefixEncoderLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, ffn_dim, dropout, max_positions):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_positions = max_positions

        self.token_emb = nn.Embedding(vocab_size,d_model)
        self.pos = SinusoidalPositionalEncoding(d_model,max_positions)
        self.drop = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            EncoderBlock(d_model,n_heads,ffn_dim,dropout)
            for _ in range(n_layers)
        ])
        self.final_ln = nn.LayerNorm(d_model)

        # Tied output projection: no second vocab-sized matrix.
        self.apply(self._init_weights)

        # GPT-style residual projection scaling improves deep-model stability.
        scaled_std = 0.02 / math.sqrt(2*n_layers)
        for block in self.blocks:
            nn.init.normal_(block.attn.out_proj.weight, mean=0.0, std=scaled_std)
            nn.init.normal_(block.ff[2].weight, mean=0.0, std=scaled_std)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    @staticmethod
    def prefix_mask(T, prefix_len, device):
        # True means attention is allowed for SDPA boolean mask.
        q = torch.arange(T, device=device)[:,None]
        k = torch.arange(T, device=device)[None,:]
        p = int(prefix_len)
        allowed = ((q < p) & (k < p)) | ((q >= p) & (k <= q))
        return allowed[None,None,:,:]

    def forward_hidden(self, input_ids, prefix_len=None, past_key_values=None, use_cache=False,
                       position_ids=None, key_valid=None):
        # position_ids / key_valid are only used for LEFT-PADDED BATCHED generation; when both are None the
        # behaviour is exactly the original (training path unchanged).
        B,T = input_ids.shape

        past_len = 0
        if past_key_values is not None:
            past_len = past_key_values[0][0].size(2)

        if past_len + T > self.max_positions:
            raise ValueError(
                f"context {past_len+T} exceeds max_positions={self.max_positions}"
            )

        if position_ids is None:
            position_ids = torch.arange(
                past_len, past_len+T, device=input_ids.device
            )[None,:].expand(B,T)

        x = self.token_emb(input_ids) + self.pos(position_ids).to(self.token_emb.weight.dtype)
        x = self.drop(x)

        # Full training sequence uses the prefix-LM mask.
        # Initial generation prefill has prefix_len == T, therefore it is fully bidirectional.
        # Cached one-token decoding needs no mask because only past/current KVs exist.
        if key_valid is not None:
            # (B,1,1,S) bool, True = real token. Prompt is bidirectional; padding is never attended to.
            attn_mask = key_valid[:,None,None,:]
        elif past_key_values is None and prefix_len is not None and int(prefix_len) < T:
            attn_mask = self.prefix_mask(T,prefix_len,input_ids.device)
        else:
            attn_mask = None

        new_cache = [] if use_cache else None

        for i,block in enumerate(self.blocks):
            past = None if past_key_values is None else past_key_values[i]
            x,kv = block(
                x,
                attn_mask=attn_mask,
                past_kv=past,
                use_cache=use_cache,
            )
            if use_cache:
                new_cache.append(kv)

        x = self.final_ln(x)
        return x,new_cache

    def project(self, hidden):
        return F.linear(hidden, self.token_emb.weight)


In [10]:
# CELL 8 — load Model1 + define continuous per-layer KV Prefix Tuning
def build_backbone():
    return PrefixEncoderLM(
        vocab_size=len(tokenizer),
        d_model=D_MODEL,
        n_layers=N_LAYERS,
        n_heads=N_HEADS,
        ffn_dim=FFN_DIM,
        dropout=DROPOUT,
        max_positions=MAX_POSITIONS,
    )

backbone = build_backbone()
phase1_obj = torch.load(
    MODEL1_CKPT, map_location="cpu", weights_only=False
)
assert isinstance(phase1_obj, dict) and "model" in phase1_obj

missing, unexpected = backbone.load_state_dict(
    phase1_obj["model"], strict=False
)
assert not missing and not unexpected, (missing, unexpected)

print(
    "✅ Model1 load PASS | %.3fM params"
    % (sum(p.numel() for p in backbone.parameters()) / 1e6)
)
print("checkpoint global_step:", phase1_obj.get("global_step"))
print("checkpoint best_val:", phase1_obj.get("best_val"))
print("checkpoint val_loss:", phase1_obj.get("val_loss"))

# Freeze all original Model1 parameters.
for p in backbone.parameters():
    p.requires_grad = False


class LayerKVPrefix(nn.Module):
    """
    Direct continuous prefix in attention-memory space.

    Each adapted layer gets trainable key/value tensors:
        K_prefix, V_prefix: [1, H, P, head_dim]

    They are concatenated before the real document K/V.
    Original Q/K/V projection weights remain frozen.
    """
    def __init__(
        self,
        n_heads=N_HEADS,
        prefix_len=PREFIX_LEN,
        head_dim=HEAD_DIM,
        init_std=PREFIX_INIT_STD,
        dropout=PREFIX_DROPOUT,
    ):
        super().__init__()
        self.key = nn.Parameter(
            torch.empty(1, n_heads, prefix_len, head_dim)
        )
        self.value = nn.Parameter(
            torch.empty(1, n_heads, prefix_len, head_dim)
        )
        nn.init.normal_(self.key, mean=0.0, std=init_std)
        nn.init.normal_(self.value, mean=0.0, std=init_std)
        self.dropout = nn.Dropout(dropout)

    def forward(self, batch_size):
        # Prefix dropout is applied during training only.
        k = self.dropout(self.key)
        v = self.dropout(self.value)
        return (
            k.expand(batch_size, -1, -1, -1),
            v.expand(batch_size, -1, -1, -1),
        )


class PrefixBank(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleDict({
            str(i): LayerKVPrefix()
            for i in PREFIX_BLOCKS
        })

    def get(self, layer_idx, batch_size):
        return self.layers[str(layer_idx)](batch_size)


prefix_bank = PrefixBank()

prefix_params = sum(p.numel() for p in prefix_bank.parameters())
print("Trainable prefix params:", prefix_params)
print(
    "Prefix params as % of backbone:",
    100 * prefix_params / sum(p.numel() for p in backbone.parameters())
)
assert prefix_params > 0


✅ Model1 load PASS | 118.106M params
checkpoint global_step: 20332
checkpoint best_val: None
checkpoint val_loss: 2.3875660663470626
Trainable prefix params: 147456
Prefix params as % of backbone: 0.12485043957758934


In [11]:
# CELL 9 — fast 3-view hierarchical Prefix-Tuning classifier
class StatutePrefixClassifier(nn.Module):
    def __init__(
        self,
        backbone,
        prefix_bank,
        d_model=768,
        proj_dim=320,
        n_labels=100,
        dropout=0.10,
    ):
        super().__init__()
        self.backbone = backbone
        self.prefix_bank = prefix_bank

        self.proj = nn.Sequential(
            nn.Linear(2 * d_model, proj_dim),
            nn.GELU(),
            nn.LayerNorm(proj_dim),
            nn.Dropout(dropout),
        )

        self.chunk_pos = nn.Parameter(
            torch.zeros(N_CHUNKS, proj_dim)
        )
        nn.init.normal_(self.chunk_pos, std=0.02)

        self.attn = nn.Sequential(
            nn.Linear(proj_dim, 96),
            nn.Tanh(),
            nn.Linear(96, 1),
        )

        self.head = nn.Sequential(
            nn.Linear(2 * proj_dim, proj_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(proj_dim, n_labels),
        )

    def encode_flat_chunks(self, ids, valid):
        B, T = ids.shape

        position_ids = torch.arange(
            T, device=ids.device
        )[None, :].expand(B, T)

        # ---- completely frozen lower 8 layers ----
        with torch.no_grad():
            x = (
                self.backbone.token_emb(ids)
                + self.backbone.pos(position_ids).to(
                    self.backbone.token_emb.weight.dtype
                )
            )
            x = self.backbone.drop(x)

            # Bidirectional document attention, but never attend to right padding.
            base_mask = valid[:, None, None, :]

            for block in self.backbone.blocks[:PREFIX_START_BLOCK]:
                x, _ = block(
                    x,
                    attn_mask=base_mask,
                    past_kv=None,
                    use_cache=False,
                )

        x = x.detach()

        # ---- top 6 layers: frozen weights + trainable KV prefixes ----
        for bi in PREFIX_BLOCKS:
            pk, pv = self.prefix_bank.get(bi, B)

            # Prefix positions are always valid, followed by document validity.
            prefix_ok = torch.ones(
                (B, PREFIX_LEN),
                dtype=torch.bool,
                device=ids.device,
            )
            kv_valid = torch.cat(
                [prefix_ok, valid], dim=1
            )
            attn_mask = kv_valid[:, None, None, :]

            x, _ = self.backbone.blocks[bi](
                x,
                attn_mask=attn_mask,
                past_kv=(pk, pv),
                use_cache=False,
            )

        x = self.backbone.final_ln(x)

        mask = valid.unsqueeze(-1).to(x.dtype)
        mean = (x * mask).sum(1) / mask.sum(1).clamp_min(1.0)

        # Model1's document representation is bidirectional in this task;
        # combine mean and last-real-token summaries.
        last_idx = valid.long().sum(1).clamp_min(1) - 1
        last = x[
            torch.arange(B, device=x.device),
            last_idx
        ]

        return self.proj(
            torch.cat([last, mean], dim=-1)
        )

    def forward(self, input_ids, valid):
        B, C, T = input_ids.shape
        z = self.encode_flat_chunks(
            input_ids.reshape(B * C, T),
            valid.reshape(B * C, T),
        ).reshape(B, C, -1)

        z = z + self.chunk_pos[None, :, :]
        scores = self.attn(z).squeeze(-1)
        weights = torch.softmax(scores, dim=-1)

        attn_pool = (
            z * weights.unsqueeze(-1)
        ).sum(1)
        max_pool = z.max(1).values

        logits = self.head(
            torch.cat([attn_pool, max_pool], dim=-1)
        )
        return logits, weights


model = StatutePrefixClassifier(
    backbone, prefix_bank
).to(DEVICE)

total = sum(p.numel() for p in model.parameters())
trainable = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)
prefix_trainable = sum(
    p.numel() for p in model.prefix_bank.parameters()
)

print(f"total={total/1e6:.3f}M")
print(
    f"trainable={trainable/1e6:.3f}M "
    f"({100*trainable/total:.3f}%)"
)
print(
    f"prefix-only trainable={prefix_trainable/1e6:.3f}M"
)

# Backbone must remain completely frozen.
assert not any(
    p.requires_grad
    for p in model.backbone.parameters()
)
assert prefix_trainable > 0


# ---- v3: warm-start the continuous prefix from REAL Model1 K/V activations ----
@torch.no_grad()
def pick_prefix_probe_ids():
    """
    Pick one full-length statute-evidence training view.
    This is used only once to initialize prefix K/V values.
    """
    evidence_lens = np.asarray(train_lens[:, 1]).astype(np.int64)
    full = np.flatnonzero(evidence_lens >= CHUNK_LEN)
    if len(full):
        # Deterministic non-first example to avoid an arbitrary edge case.
        idx = int(full[min(7, len(full) - 1)])
    else:
        idx = int(np.argmax(evidence_lens))

    L = int(evidence_lens[idx])
    ids = np.array(
        train_tok[idx, 1, :L],
        dtype=np.int64,
        copy=True,
    )
    print(
        "prefix-init probe row:", idx,
        "| evidence tokens:", L,
    )
    return torch.from_numpy(ids)


@torch.no_grad()
def init_prefix_from_real_tokens(clf, probe_ids):
    """
    Run the frozen pretrained Model1 on one real document chunk and
    initialize each adapted layer's trainable prefix with K/V vectors
    generated by real hidden states at that same layer.

    The prefix mechanism itself is unchanged; only its initialization differs.
    """
    was_training = clf.training
    clf.eval()

    bb = clf.backbone
    bank = clf.prefix_bank

    ids = probe_ids[None, :].to(DEVICE)
    T = ids.shape[1]
    assert T >= PREFIX_LEN, (T, PREFIX_LEN)

    pos = torch.arange(
        T, device=DEVICE
    )[None, :]

    x = (
        bb.token_emb(ids)
        + bb.pos(pos).to(
            bb.token_emb.weight.dtype
        )
    )
    x = bb.drop(x)

    # Evenly-spaced real positions give broad coverage of the probe chunk.
    sel = torch.linspace(
        0, T - 1,
        PREFIX_LEN,
        device=DEVICE,
    ).long()

    for bi, block in enumerate(bb.blocks):
        if bi in PREFIX_BLOCKS:
            h = block.ln1(x)
            qkv = block.attn.qkv(h).view(
                1, T, 3,
                N_HEADS, HEAD_DIM
            )
            _, k, v = qkv.unbind(dim=2)
            k = k.transpose(1, 2)  # [1,H,T,hd]
            v = v.transpose(1, 2)

            layer_prefix = bank.layers[str(bi)]
            assert layer_prefix.key.shape == (
                1, N_HEADS, PREFIX_LEN, HEAD_DIM
            )
            layer_prefix.key.copy_(
                k[:, :, sel, :].to(
                    layer_prefix.key.dtype
                )
            )
            layer_prefix.value.copy_(
                v[:, :, sel, :].to(
                    layer_prefix.value.dtype
                )
            )

        # Continue through the ORIGINAL frozen backbone, without prefixes,
        # only to obtain realistic deeper-layer states for initialization.
        x, _ = block(
            x,
            attn_mask=None,
            past_kv=None,
            use_cache=False,
        )

    clf.train(was_training)


PROBE_IDS = pick_prefix_probe_ids()

if PREFIX_INIT_FROM_REAL_TOKENS:
    try:
        init_prefix_from_real_tokens(
            model, PROBE_IDS
        )
        key_stds = [
            model.prefix_bank.layers[str(i)]
            .key.detach().float().std().item()
            for i in PREFIX_BLOCKS
        ]
        value_stds = [
            model.prefix_bank.layers[str(i)]
            .value.detach().float().std().item()
            for i in PREFIX_BLOCKS
        ]
        print(
            "✅ prefix initialized from real-token K/V"
        )
        print(
            "   key std per layer:",
            [round(x, 4) for x in key_stds]
        )
        print(
            "   val std per layer:",
            [round(x, 4) for x in value_stds]
        )
    except Exception as e:
        print(
            "⚠️ real-token prefix init failed; "
            "using original Gaussian init:",
            repr(e)[:250],
        )


total=119.015M
trainable=0.909M (0.764%)
prefix-only trainable=0.147M
prefix-init probe row: 9 | evidence tokens: 320
✅ prefix initialized from real-token K/V
   key std per layer: [3.2487, 5.2844, 3.5281, 3.1013, 1.5763, 2.4367]
   val std per layer: [0.8464, 1.0675, 1.0368, 1.3714, 1.5418, 1.59]


In [12]:
# CELL 10 — datasets/loaders
class LSIDataset(Dataset):
    def __init__(self,records,tok,lens,labeled=True):
        self.records=records; self.tok=tok; self.lens=lens; self.labeled=labeled
    def __len__(self): return len(self.records)
    def __getitem__(self,i):
        ids=torch.from_numpy(np.array(self.tok[i],dtype=np.int64,copy=True))
        lens=torch.from_numpy(np.array(self.lens[i],dtype=np.int64,copy=True))
        valid=torch.arange(CHUNK_LEN)[None,:]<lens[:,None]
        out={"input_ids":ids,"valid":valid,"id":str(self.records[i]["id"])}
        if self.labeled:
            y=torch.zeros(N_LABELS,dtype=torch.float32)
            y[self.records[i]["labels"]]=1.0
            out["labels"]=y
        return out

def make_loader(records,tok,lens,batch,shuffle,labeled=True,seed=SEED):
    # v2: persistent workers only for the training loader (dev/test loaders are rebuilt cheaply & survive interrupts)
    g=torch.Generator(); g.manual_seed(seed)
    return DataLoader(
        LSIDataset(records,tok,lens,labeled),
        batch_size=batch,shuffle=shuffle,num_workers=NUM_WORKERS,
        pin_memory=True,persistent_workers=(NUM_WORKERS>0 and shuffle),
        generator=g
    )

train_loader=make_loader(train_records,train_tok,train_lens,MICRO_BATCH_SIZE,True,True)
official_loader=make_loader(dev_raw,official_tok,official_lens,DEV_BATCH_SIZE,False,True)
test_loader=make_loader(test_records,test_tok,test_lens,DEV_BATCH_SIZE,False,False)

print("loader batches:",len(train_loader),len(official_loader),len(test_loader))
print("train tokens/microbatch:",MICRO_BATCH_SIZE*N_CHUNKS*CHUNK_LEN)


loader batches: 10688 637 814
train tokens/microbatch: 3840


In [13]:
# CELL 11 — exact evaluator logic + threshold tuning
def multilabel_metrics(y_true,y_pred):
    return {
        "macro_f1":float(f1_score(y_true,y_pred,average="macro",zero_division=0)),
        "micro_f1":float(f1_score(y_true,y_pred,average="micro",zero_division=0)),
        "subset_accuracy":float(accuracy_score(y_true,y_pred)),
    }

def apply_thresholds(prob,thresholds,ensure_nonempty=True):
    thr=np.asarray(thresholds,dtype=np.float32)
    pred=prob>=thr[None,:]
    if ensure_nonempty:
        empty=np.flatnonzero(~pred.any(1))
        if len(empty):
            best=np.argmax(prob[empty]/np.maximum(thr[None,:],1e-6),axis=1)
            pred[empty,best]=True
    return pred

def macro_f1_fast(y,pred):
    yb=y.astype(bool); pb=pred.astype(bool)
    tp=(yb&pb).sum(0).astype(np.float64)
    den=yb.sum(0)+pb.sum(0)
    return float(np.where(den>0,2*tp/np.maximum(den,1),0.0).mean())

def f1_curves(y,prob,grid=THR_GRID):
    yb=y.astype(bool); P=yb.sum(0).astype(np.float64)
    out=np.zeros((len(grid),y.shape[1]),dtype=np.float64)
    for gi,t in enumerate(grid):
        pr=prob>=t
        tp=(pr&yb).sum(0); den=pr.sum(0)+P
        out[gi]=np.where(den>0,2*tp/np.maximum(den,1),0.0)
    return out

def smooth_curves(c):
    p=np.pad(c,((1,1),(0,0)),mode="edge")
    return (p[:-2]+p[1:-1]+p[2:])/3.0

def fit_threshold_strategies(y,prob):
    c=f1_curves(y,prob)
    g=float(THR_GRID[int(np.argmax(c.mean(1)))])
    per=THR_GRID[np.argmax(smooth_curves(c),axis=0)].astype(np.float32)
    P=y.sum(0).astype(np.float64)
    out={"global":np.full(y.shape[1],g,dtype=np.float32),"per_label":per}
    for tau in SHRINK_TAUS:
        w=P/(P+tau)
        out[f"shrunk_tau_{int(tau)}"]=(w*per+(1-w)*g).astype(np.float32)
    return out,g,per

def optimize_thresholds(y,prob):
    full,g,per=fit_threshold_strategies(y,prob)
    n=len(y); cv={k:[] for k in full}
    for rep_i in range(CV_REPEATS):
        perm=np.random.RandomState(100+rep_i).permutation(n)
        a,b=perm[:n//2],perm[n//2:]
        for tr,te in ((a,b),(b,a)):
            strat,_,_=fit_threshold_strategies(y[tr],prob[tr])
            for k,thr in strat.items():
                cv[k].append(
                    macro_f1_fast(y[te],apply_thresholds(prob[te],thr,True))
                )
    cv={k:float(np.mean(v)) for k,v in cv.items()}
    name=max(cv,key=cv.get); thr=full[name]
    return {
        "name":name,"thresholds":thr,"cv_macro_f1":cv[name],
        "insample_macro_f1":macro_f1_fast(y,apply_thresholds(prob,thr,True)),
        "cv_all":cv
    },g,per

@torch.inference_mode()
def predict_loader(model,loader):
    model.eval()
    ids=[]; probs=[]; ys=[]; atts=[]
    for b in loader:
        x=b["input_ids"].to(DEVICE,non_blocking=True)
        v=b["valid"].to(DEVICE,non_blocking=True)
        with torch.amp.autocast(
            "cuda",dtype=torch.float16,
            enabled=(DEVICE.type=="cuda" and USE_AMP)
        ):
            lg,a=model(x,v)
        assert torch.isfinite(lg).all()
        ids.extend(list(b["id"]))
        probs.append(torch.sigmoid(lg.float()).cpu().numpy())
        atts.append(a.float().cpu().numpy())
        if "labels" in b: ys.append(b["labels"].numpy().astype(np.uint8))
    return ids,np.concatenate(probs),(np.concatenate(ys) if ys else None),np.concatenate(atts)

def make_pred_records(ids,prob,thresholds,ensure_nonempty=True):
    binary=apply_thresholds(prob,thresholds,ensure_nonempty)
    return [{"id":str(i),"pred":[int(x) for x in np.flatnonzero(binary[k])]} for k,i in enumerate(ids)]

def exact_phase5_score(truth_records,pred_records):
    t={}; max_label=-1
    for r in truth_records:
        assert r["id"] not in t
        seen=set()
        for label in r["gold"]:
            assert isinstance(label,int) and not isinstance(label,bool) and label>=0 and label not in seen
            seen.add(label); max_label=max(max_label,label)
        t[r["id"]]=r["gold"]

    nlab=max_label+1
    p={}
    for r in pred_records:
        assert r["id"] not in p
        seen=set()
        for x in r["pred"]:
            assert isinstance(x,int) and not isinstance(x,bool) and 0<=x<nlab and x not in seen
            seen.add(x)
        p[r["id"]]=r["pred"]

    assert set(t)==set(p)
    yt=np.zeros((len(t),nlab),dtype=np.uint8)
    yp=np.zeros((len(t),nlab),dtype=np.uint8)
    for i,k in enumerate(t):
        for x in t[k]: yt[i,x]=1
        for x in p[k]: yp[i,x]=1
    return multilabel_metrics(yt,yp)

print("✅ metric/threshold utilities ready")


✅ metric/threshold utilities ready


In [14]:
# CELL 12 — Prefix-Tuning training: tiny trainable set + ONE DEV pass/epoch
pos_weight_t = torch.tensor(
    POS_WEIGHT,
    dtype=torch.float32,
    device=DEVICE,
)

prefix_params = list(
    model.prefix_bank.parameters()
)
head_params = [
    p for n, p in model.named_parameters()
    if p.requires_grad
    and not n.startswith("prefix_bank.")
]

assert prefix_params and head_params

print(
    "Prefix trainable:",
    sum(p.numel() for p in prefix_params)
)
print(
    "Classifier trainable:",
    sum(p.numel() for p in head_params)
)

# Fused AdamW where available.
optim_groups = [
    {
        "params": prefix_params,
        "lr": PREFIX_LR,
        "weight_decay": 0.0,
    },
    {
        "params": head_params,
        "lr": HEAD_LR,
        "weight_decay": WEIGHT_DECAY,
    },
]

try:
    optimizer = torch.optim.AdamW(
        optim_groups,
        betas=(0.9, 0.999),
        eps=1e-8,
        fused=(DEVICE.type == "cuda"),
    )
    print("AdamW fused:", DEVICE.type == "cuda")
except Exception:
    optimizer = torch.optim.AdamW(
        optim_groups,
        betas=(0.9, 0.999),
        eps=1e-8,
    )
    print("Using standard AdamW")

updates_per_epoch = math.ceil(
    len(train_loader) / GRAD_ACCUM_STEPS
)
total_updates = updates_per_epoch * EPOCHS
warmup = max(
    1, int(WARMUP_RATIO * total_updates)
)

def lr_lambda(step):
    if step < warmup:
        return (step + 1) / warmup
    q = (
        step - warmup
    ) / max(1, total_updates - warmup)
    return max(
        0.08,
        0.5 * (
            1 + math.cos(
                math.pi * min(1.0, q)
            )
        ),
    )

scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lr_lambda
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(
        DEVICE.type == "cuda" and USE_AMP
    ),
)

BEST = ROOT / "best_exp4_prefix.pt"
HISTORY = ROOT / "history.json"


class EMA:
    def __init__(self, model, decay):
        self.decay = decay
        self.params = {
            n: p
            for n, p in model.named_parameters()
            if p.requires_grad
        }
        self.shadow = {
            n: p.detach().clone()
            for n, p in self.params.items()
        }
        self.n = 0

    @torch.no_grad()
    def update(self):
        self.n += 1
        d = min(
            self.decay,
            (1.0 + self.n) / (10.0 + self.n),
        )
        for n, p in self.params.items():
            self.shadow[n].mul_(d).add_(
                p.detach(), alpha=1.0-d
            )

    @contextlib.contextmanager
    def applied(self):
        backup = {
            n: p.detach().clone()
            for n, p in self.params.items()
        }
        with torch.no_grad():
            for n, p in self.params.items():
                p.copy_(self.shadow[n])
        try:
            yield
        finally:
            with torch.no_grad():
                for n, p in self.params.items():
                    p.copy_(backup[n])


def get_trainable_state(model):
    return {
        n: p.detach().cpu().clone()
        for n, p in model.named_parameters()
        if p.requires_grad
    }


def load_trainable_state(model, state):
    named = dict(model.named_parameters())
    with torch.no_grad():
        for n, v in state.items():
            assert n in named, n
            named[n].copy_(
                v.to(
                    named[n].device,
                    dtype=named[n].dtype,
                )
            )


ema = EMA(model, EMA_DECAY)
best_score = -1.0
history = []
global_update = 0
optimizer.zero_grad(set_to_none=True)


def evaluate_epoch(epoch, tag="ema"):
    global best_score

    oids, oprob, oy, oatt = predict_loader(
        model, official_loader
    )

    cprob = oprob[CLEAN_IDX]
    cy = oy[CLEAN_IDX]
    cids = [oids[i] for i in CLEAN_IDX]
    catt = oatt[CLEAN_IDX]

    tune, g, per = optimize_thresholds(
        cy, cprob
    )

    clean_pred = make_pred_records(
        cids,
        cprob,
        tune["thresholds"],
        True,
    )
    official_pred = make_pred_records(
        oids,
        oprob,
        tune["thresholds"],
        True,
    )

    exact_clean = exact_phase5_score(
        clean_truth, clean_pred
    )
    exact_official = exact_phase5_score(
        official_truth, official_pred
    )

    row = {
        "epoch": epoch,
        "weights": tag,
        "strategy": tune["name"],
        "global_threshold": g,
        "cv_macro_f1": tune["cv_macro_f1"],
        "insample_macro_f1": tune[
            "insample_macro_f1"
        ],
        "cv_all": tune["cv_all"],
        "clean": exact_clean,
        "official": exact_official,
        "mean_pred_labels_clean": float(
            np.mean(
                [
                    len(r["pred"])
                    for r in clean_pred
                ]
            )
        ),
        "mean_gold_labels_clean": float(
            cy.sum(1).mean()
        ),
        "attention_mean_clean": catt.mean(
            0
        ).tolist(),
    }

    history.append(row)
    json.dump(
        history,
        open(HISTORY, "w"),
        indent=2,
    )

    print(
        f"[{tag}] epoch={epoch} "
        f"strat={tune['name']} "
        f"CV-macroF1={tune['cv_macro_f1']:.4f} | "
        f"clean={exact_clean['macro_f1']:.4f}/"
        f"{exact_clean['micro_f1']:.4f}/"
        f"{exact_clean['subset_accuracy']:.4f} | "
        f"official-macro={exact_official['macro_f1']:.4f} | "
        f"pred/gold="
        f"{row['mean_pred_labels_clean']:.2f}/"
        f"{row['mean_gold_labels_clean']:.2f}",
        flush=True,
    )

    if tune["cv_macro_f1"] > best_score:
        best_score = tune["cv_macro_f1"]
        atomic_torch_save(
            {
                "trainable_state":
                    get_trainable_state(model),
                "thresholds": np.asarray(
                    tune["thresholds"],
                    dtype=np.float32,
                ),
                "threshold_strategy":
                    tune["name"],
                "global_threshold": g,
                "per_label_thresholds":
                    np.asarray(
                        per,
                        dtype=np.float32,
                    ),
                "cv_macro_f1":
                    tune["cv_macro_f1"],
                "clean_metrics":
                    exact_clean,
                "official_metrics":
                    exact_official,
                "epoch": epoch,
                "weights_tag": tag,
                "model1_sha256":
                    sha256_file(MODEL1_CKPT),
                "tokenizer_sha256":
                    sha256_file(
                        TOKENIZER_FILE
                    ),
                "dataset_sha256":
                    DATA_SHA256,
                "config": {
                    "method":
                        "Prefix Tuning",
                    "prefix_len":
                        PREFIX_LEN,
                    "prefix_start_block":
                        PREFIX_START_BLOCK,
                    "prefix_blocks":
                        PREFIX_BLOCKS,
                    "prefix_lr":
                        PREFIX_LR,
                    "head_lr":
                        HEAD_LR,
                    "n_chunks":
                        N_CHUNKS,
                    "chunk_len":
                        CHUNK_LEN,
                    "views":
                        VIEW_NAMES,
                    "epochs":
                        EPOCHS,
                    "pos_weight":
                        "sqrt(raw) clipped [1,10]",
                },
            },
            BEST,
        )
        print(
            "✅ NEW BEST:",
            round(best_score, 5),
        )


print("batches/epoch:", len(train_loader))
print("optimizer updates:", total_updates)
print(
    "Backbone trainable params:",
    sum(
        p.numel()
        for p in model.backbone.parameters()
        if p.requires_grad
    ),
)
print(
    "OOM fallback: MICRO_BATCH_SIZE=2, "
    "GRAD_ACCUM_STEPS=8."
)

for epoch in range(1, EPOCHS + 1):
    model.train()
    t0 = time.time()
    running = 0.0
    good = 0

    for bi, b in enumerate(
        train_loader, 1
    ):
        x = b["input_ids"].to(
            DEVICE, non_blocking=True
        )
        v = b["valid"].to(
            DEVICE, non_blocking=True
        )
        y = b["labels"].to(
            DEVICE, non_blocking=True
        )

        with torch.amp.autocast(
            "cuda",
            dtype=torch.float16,
            enabled=(
                DEVICE.type == "cuda"
                and USE_AMP
            ),
        ):
            logits, _ = model(x, v)
            loss = (
                F.binary_cross_entropy_with_logits(
                    logits,
                    y,
                    pos_weight=pos_weight_t,
                )
            )
            scaled_loss = (
                loss / GRAD_ACCUM_STEPS
            )

        if torch.isfinite(loss):
            scaler.scale(
                scaled_loss
            ).backward()
            good += 1
            running += float(
                loss.detach()
            )
        else:
            print(
                "⚠️ non-finite loss batch",
                bi,
            )

        if (
            bi % GRAD_ACCUM_STEPS == 0
            or bi == len(train_loader)
        ):
            if good:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    [
                        p
                        for p in model.parameters()
                        if p.requires_grad
                    ],
                    GRAD_CLIP,
                )

                old_scale = float(
                    scaler.get_scale()
                )
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(
                    set_to_none=True
                )

                if (
                    float(
                        scaler.get_scale()
                    )
                    >= old_scale
                ):
                    scheduler.step()
                    global_update += 1
                    ema.update()
            else:
                optimizer.zero_grad(
                    set_to_none=True
                )
            good = 0

        if (
            bi % 500 == 0
            or bi == len(train_loader)
        ):
            elapsed = (
                time.time() - t0
            ) / 60
            eta = (
                elapsed
                * (
                    len(train_loader) - bi
                )
                / max(1, bi)
            )
            print(
                f"epoch={epoch} "
                f"batch={bi}/{len(train_loader)} "
                f"loss={running/max(1,bi):.4f} "
                f"elapsed={elapsed:.1f}m "
                f"ETA={eta:.1f}m",
                flush=True,
            )

    # Only one DEV inference per epoch.
    with ema.applied():
        evaluate_epoch(epoch, "ema")

    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

assert BEST.exists()
print(
    "✅ Prefix-Tuning complete | "
    "best clean-DEV CV Macro-F1:",
    best_score,
)


Prefix trainable: 147456
Classifier trainable: 761573
AdamW fused: True
batches/epoch: 10688
optimizer updates: 5344
Backbone trainable params: 0
OOM fallback: MICRO_BATCH_SIZE=2, GRAD_ACCUM_STEPS=8.
epoch=1 batch=500/10688 loss=0.5394 elapsed=0.7m ETA=14.8m
epoch=1 batch=1000/10688 loss=0.4949 elapsed=1.5m ETA=14.2m
epoch=1 batch=1500/10688 loss=0.4772 elapsed=2.2m ETA=13.7m
epoch=1 batch=2000/10688 loss=0.4664 elapsed=3.0m ETA=13.1m
epoch=1 batch=2500/10688 loss=0.4578 elapsed=3.8m ETA=12.4m
epoch=1 batch=3000/10688 loss=0.4502 elapsed=4.6m ETA=11.7m
epoch=1 batch=3500/10688 loss=0.4438 elapsed=5.3m ETA=11.0m
epoch=1 batch=4000/10688 loss=0.4391 elapsed=6.1m ETA=10.2m
epoch=1 batch=4500/10688 loss=0.4357 elapsed=6.9m ETA=9.5m
epoch=1 batch=5000/10688 loss=0.4308 elapsed=7.7m ETA=8.7m
epoch=1 batch=5500/10688 loss=0.4268 elapsed=8.5m ETA=8.0m
epoch=1 batch=6000/10688 loss=0.4247 elapsed=9.2m ETA=7.2m
epoch=1 batch=6500/10688 loss=0.4218 elapsed=10.0m ETA=6.4m
epoch=1 batch=7000/10688 

In [15]:
# CELL 13 — reload BEST Prefix-Tuning state + DEV diagnostics
best = torch.load(
    BEST,
    map_location="cpu",
    weights_only=False,
)
load_trainable_state(
    model, best["trainable_state"]
)
model.to(DEVICE)
model.eval()

thresholds = np.asarray(
    best["thresholds"],
    dtype=np.float32,
)

print("BEST epoch:", best["epoch"])
print(
    "threshold strategy:",
    best["threshold_strategy"],
)
print(
    "CV macro-F1:",
    best["cv_macro_f1"],
)
print(
    "clean metrics:",
    best["clean_metrics"],
)
print(
    "official metrics:",
    best["official_metrics"],
)
print(
    "threshold min/median/max:",
    thresholds.min(),
    np.median(thresholds),
    thresholds.max(),
)

dids, dprob, dy, datt = predict_loader(
    model, official_loader
)

cids = [dids[i] for i in CLEAN_IDX]
cprob = dprob[CLEAN_IDX]

re_pred = make_pred_records(
    cids,
    cprob,
    thresholds,
    True,
)
re_score = exact_phase5_score(
    clean_truth, re_pred
)
print(
    "re-scored clean DEV:",
    re_score,
)

assert abs(
    re_score["macro_f1"]
    - best["clean_metrics"]["macro_f1"]
) < 5e-3

print(
    "mean attention "
    "[HEAD,EVIDENCE,TAIL]:",
    datt[CLEAN_IDX].mean(0),
)

np.savez_compressed(
    ROOT / "dev_probs_exp4.npz",
    id=np.asarray(dids),
    prob=dprob,
    y=dy,
    clean_idx=CLEAN_IDX,
    thresholds=thresholds,
)


BEST epoch: 2
threshold strategy: per_label
CV macro-F1: 0.2117679947567438
clean metrics: {'macro_f1': 0.2236957878577667, 'micro_f1': 0.30938307129608766, 'subset_accuracy': 0.0013912352181258073}
official metrics: {'macro_f1': 0.22323237312445623, 'micro_f1': 0.30938640813220825, 'subset_accuracy': 0.001375110499950889}
threshold min/median/max: 0.17 0.37 0.78
re-scored clean DEV: {'macro_f1': 0.2236957878577667, 'micro_f1': 0.30938307129608766, 'subset_accuracy': 0.0013912352181258073}
mean attention [HEAD,EVIDENCE,TAIL]: [0.40202036 0.23699427 0.36098462]


In [16]:
# CELL 14 — exact official Phase-5 evaluator + literal DEV script test
EVAL_SCRIPT = ROOT / "evaluate_phase5.py"
EVAL_SCRIPT.write_text('#!/usr/bin/env python3\n"""Local evaluator for Phase 5.\n\nTruth JSON: a list of objects, [{"id":"...", "gold":[0,2,5,...]}, ...]\nPrediction JSON: [{"id":"...", "pred":[0,2,5,...]}, ...]\n"""\nimport argparse,json,sys\nimport numpy as np\nfrom sklearn.metrics import accuracy_score, f1_score\n\ndef read_truth(path):\n    try:\n        x=json.load(open(path,encoding="utf-8"))\n    except json.JSONDecodeError as e:\n        raise ValueError(f"Truth must be a JSON file containing a list of records: {e}")\n    if not isinstance(x,list) or not x: raise ValueError(\'Truth must be a non-empty JSON list.\')\n    out={}; max_label=-1\n    for n,r in enumerate(x):\n        if not isinstance(r,dict) or \'id\' not in r or \'gold\' not in r: raise ValueError(f\'Truth item {n} must contain id and gold.\')\n        if not isinstance(r[\'id\'],str) or not r[\'id\'].strip() or not isinstance(r[\'gold\'],list): raise ValueError(f\'Invalid truth item {n}.\')\n        if r[\'id\'] in out: raise ValueError(f\'Duplicate truth ID: {r["id"]}\')\n        seen=set()\n        for label in r[\'gold\']:\n            if isinstance(label,bool) or not isinstance(label,int) or label<0 or label in seen: raise ValueError(f\'Invalid/duplicate label in truth item {n}.\')\n            seen.add(label); max_label=max(max_label,label)\n        out[r[\'id\']]=r[\'gold\']\n    return out,max_label+1\n\ndef read_pred(path,nlab):\n    x=json.load(open(path,encoding=\'utf-8\'))\n    if not isinstance(x,list) or not x: raise ValueError(\'Prediction must be a non-empty JSON list.\')\n    out={}\n    for n,r in enumerate(x):\n        if not isinstance(r,dict) or \'id\' not in r or \'pred\' not in r: raise ValueError(f\'Prediction item {n} must contain id and pred.\')\n        if not isinstance(r[\'id\'],str) or not isinstance(r[\'pred\'],list) or r[\'id\'] in out: raise ValueError(f\'Invalid or duplicate prediction item {n}.\')\n        seen=set()\n        for x in r[\'pred\']:\n            if isinstance(x,bool) or not isinstance(x,int) or x<0 or x>=nlab or x in seen: raise ValueError(f\'Invalid/duplicate label in prediction item {n}.\')\n            seen.add(x)\n        out[r[\'id\']]=r[\'pred\']\n    return out\n\ndef main():\n    ap=argparse.ArgumentParser(description=\'Evaluate Phase 5 multilabel predictions on your own held-out split.\')\n    ap.add_argument(\'--truth\',required=True); ap.add_argument(\'--prediction\',required=True); a=ap.parse_args()\n    t,n=read_truth(a.truth); p=read_pred(a.prediction,n)\n    if set(t)!=set(p): raise ValueError(f"ID mismatch: missing={len(set(t)-set(p))}, extra={len(set(p)-set(t))}")\n    yt=np.zeros((len(t),n),dtype=np.uint8); yp=np.zeros((len(t),n),dtype=np.uint8)\n    for i,k in enumerate(t):\n        for x in t[k]: yt[i,x]=1\n        for x in p[k]: yp[i,x]=1\n    print(f\'Examples: {len(t)}\'); print(f\'Labels: {n}\')\n    print(f\'Macro-F1: {f1_score(yt,yp,average="macro",zero_division=0):.6f}\')\n    micro=f1_score(yt,yp,average=\'micro\',zero_division=0)\n    subset=accuracy_score(yt,yp)\n    print(f\'Micro-F1: {micro:.6f}\')\n    print(f\'Subset Accuracy: {subset:.6f}\')\nif __name__==\'__main__\':\n    try: main()\n    except Exception as e: print(f\'EVALUATION FAILED: {e}\',file=sys.stderr); sys.exit(1)\n', encoding="utf-8")
print("✅ wrote exact evaluator:", EVAL_SCRIPT)


✅ wrote exact evaluator: /kaggle/working/phase5_exp4_model1_prefix_fast_v3/evaluate_phase5.py


In [17]:
# CELL 15 — HELDOUT TEST inference + ZIP + exact evaluator-parser validation
tids, tprob, _, tatt = predict_loader(
    model, test_loader
)

pred_records = make_pred_records(
    tids,
    tprob,
    thresholds,
    True,
)

assert len(pred_records) == len(test_records)
assert len(
    {r["id"] for r in pred_records}
) == len(test_records)

PRED = ROOT / "prediction.json"
json.dump(
    pred_records,
    open(PRED, "w", encoding="utf-8"),
    ensure_ascii=False,
)

ZIP = ROOT / "submission_exp4_Model1_Prefix_HELDOUT.zip"
with zipfile.ZipFile(
    ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as z:
    z.write(
        PRED,
        arcname="prediction.json",
    )

np.savez_compressed(
    ROOT / "test_probs_heldout_exp4.npz",
    id=np.asarray(tids),
    prob=tprob,
    thresholds=thresholds,
)

spec = importlib.util.spec_from_file_location(
    "evaluate_phase5_exact",
    EVAL_SCRIPT,
)
eval_exact = importlib.util.module_from_spec(
    spec
)
spec.loader.exec_module(eval_exact)

check_dir = ROOT / "zip_check_heldout"
if check_dir.exists():
    shutil.rmtree(check_dir)
check_dir.mkdir(
    parents=True,
    exist_ok=True,
)

with zipfile.ZipFile(
    ZIP, "r"
) as z:
    assert z.namelist() == [
        "prediction.json"
    ]
    z.extract(
        "prediction.json",
        check_dir,
    )

zip_pred = eval_exact.read_pred(
    str(
        check_dir
        / "prediction.json"
    ),
    N_LABELS,
)
expected_ids = {
    str(r["id"])
    for r in test_records
}
assert set(zip_pred) == expected_ids

print(
    "TEST mean labels/doc:",
    np.mean(
        [
            len(r["pred"])
            for r in pred_records
        ]
    ),
)
print(
    "TEST attention mean "
    "[HEAD,EVIDENCE,TAIL]:",
    tatt.mean(0),
)
print(
    "✅ HELDOUT ZIP PASS:",
    ZIP,
)


TEST mean labels/doc: 7.370612182195253
TEST attention mean [HEAD,EVIDENCE,TAIL]: [0.40024975 0.23634036 0.36341074]
✅ HELDOUT ZIP PASS: /kaggle/working/phase5_exp4_model1_prefix_fast_v3/submission_exp4_Model1_Prefix_HELDOUT.zip


## Fresh TRAIN+DEV Prefix-Tuning refit

The refit starts again from the **original Phase-1 Model1 checkpoint** with fresh prefix tensors and a fresh classifier.

It trains for the number of epochs selected by the honest heldout run, performs no DEV evaluation during refit, and reuses heldout-selected thresholds.


In [18]:
# CELL 16 — fresh TRAIN+DEV Prefix-Tuning refit
if not RUN_REFIT:
    print(
        "RUN_REFIT=False -> "
        "fresh refit skipped."
    )
else:
    REFIT_EPOCHS = max(
        1, int(best["epoch"])
    )
    REFIT_SEED = SEED + 777

    print(
        "Fresh refit epochs:",
        REFIT_EPOCHS,
    )
    print(
        "TRAIN+DEV rows:",
        len(train_records)
        + len(dev_raw),
    )

    torch.manual_seed(REFIT_SEED)
    np.random.seed(REFIT_SEED)
    random.seed(REFIT_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            REFIT_SEED
        )

    refit_backbone = build_backbone()
    obj_refit = torch.load(
        MODEL1_CKPT,
        map_location="cpu",
        weights_only=False,
    )
    missing, unexpected = (
        refit_backbone.load_state_dict(
            obj_refit["model"],
            strict=False,
        )
    )
    assert not missing and not unexpected

    for p in refit_backbone.parameters():
        p.requires_grad = False

    refit_prefix_bank = PrefixBank()
    refit_model = (
        StatutePrefixClassifier(
            refit_backbone,
            refit_prefix_bank,
        ).to(DEVICE)
    )

    if PREFIX_INIT_FROM_REAL_TOKENS:
        init_prefix_from_real_tokens(
            refit_model,
            PROBE_IDS,
        )
        print(
            "✅ refit prefix initialized "
            "from the same real-token K/V probe"
        )

    refit_prefix_params = list(
        refit_model.prefix_bank.parameters()
    )
    refit_head_params = [
        p
        for n, p
        in refit_model.named_parameters()
        if p.requires_grad
        and not n.startswith(
            "prefix_bank."
        )
    ]

    print(
        "refit prefix trainable:",
        sum(
            p.numel()
            for p in refit_prefix_params
        ),
    )
    print(
        "refit head trainable:",
        sum(
            p.numel()
            for p in refit_head_params
        ),
    )

    refit_train_ds = (
        torch.utils.data.ConcatDataset(
            [
                LSIDataset(
                    train_records,
                    train_tok,
                    train_lens,
                    labeled=True,
                ),
                LSIDataset(
                    dev_raw,
                    official_tok,
                    official_lens,
                    labeled=True,
                ),
            ]
        )
    )

    refit_gen = torch.Generator()
    refit_gen.manual_seed(
        REFIT_SEED
    )

    refit_loader = DataLoader(
        refit_train_ds,
        batch_size=MICRO_BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(
            NUM_WORKERS > 0
        ),
        generator=refit_gen,
    )

    full_counts = np.zeros(
        N_LABELS,
        dtype=np.int64,
    )
    for r in train_records:
        full_counts[
            r["labels"]
        ] += 1
    for r in dev_raw:
        full_counts[
            r["labels"]
        ] += 1

    full_n = (
        len(train_records)
        + len(dev_raw)
    )
    full_raw_pw = (
        full_n - full_counts
    ) / full_counts

    refit_pos_weight_t = (
        torch.tensor(
            np.clip(
                np.sqrt(full_raw_pw),
                1.0,
                10.0,
            ).astype(np.float32),
            dtype=torch.float32,
            device=DEVICE,
        )
    )

    refit_groups = [
        {
            "params":
                refit_prefix_params,
            "lr": PREFIX_LR,
            "weight_decay": 0.0,
        },
        {
            "params":
                refit_head_params,
            "lr": HEAD_LR,
            "weight_decay":
                WEIGHT_DECAY,
        },
    ]

    try:
        refit_optimizer = (
            torch.optim.AdamW(
                refit_groups,
                betas=(0.9, 0.999),
                eps=1e-8,
                fused=(
                    DEVICE.type
                    == "cuda"
                ),
            )
        )
    except Exception:
        refit_optimizer = (
            torch.optim.AdamW(
                refit_groups,
                betas=(0.9, 0.999),
                eps=1e-8,
            )
        )

    refit_updates_per_epoch = (
        math.ceil(
            len(refit_loader)
            / GRAD_ACCUM_STEPS
        )
    )
    refit_total_updates = (
        refit_updates_per_epoch
        * REFIT_EPOCHS
    )
    refit_warmup = max(
        1,
        int(
            WARMUP_RATIO
            * refit_total_updates
        ),
    )

    def refit_lr_lambda(step):
        if step < refit_warmup:
            return (
                step + 1
            ) / refit_warmup

        q = (
            step - refit_warmup
        ) / max(
            1,
            refit_total_updates
            - refit_warmup,
        )
        return max(
            0.08,
            0.5
            * (
                1
                + math.cos(
                    math.pi
                    * min(1.0, q)
                )
            ),
        )

    refit_scheduler = (
        torch.optim.lr_scheduler.LambdaLR(
            refit_optimizer,
            refit_lr_lambda,
        )
    )

    refit_scaler = (
        torch.amp.GradScaler(
            "cuda",
            enabled=(
                DEVICE.type == "cuda"
                and USE_AMP
            ),
        )
    )
    refit_ema = EMA(
        refit_model,
        EMA_DECAY,
    )

    refit_optimizer.zero_grad(
        set_to_none=True
    )
    refit_global_update = 0

    for epoch in range(
        1, REFIT_EPOCHS + 1
    ):
        refit_model.train()
        t0 = time.time()
        running = 0.0
        good = 0

        for bi, b in enumerate(
            refit_loader, 1
        ):
            x = b[
                "input_ids"
            ].to(
                DEVICE,
                non_blocking=True,
            )
            v = b["valid"].to(
                DEVICE,
                non_blocking=True,
            )
            y = b["labels"].to(
                DEVICE,
                non_blocking=True,
            )

            with torch.amp.autocast(
                "cuda",
                dtype=torch.float16,
                enabled=(
                    DEVICE.type
                    == "cuda"
                    and USE_AMP
                ),
            ):
                logits, _ = (
                    refit_model(x, v)
                )
                loss = (
                    F.binary_cross_entropy_with_logits(
                        logits,
                        y,
                        pos_weight=
                            refit_pos_weight_t,
                    )
                )
                scaled_loss = (
                    loss
                    / GRAD_ACCUM_STEPS
                )

            if torch.isfinite(loss):
                refit_scaler.scale(
                    scaled_loss
                ).backward()
                running += float(
                    loss.detach()
                )
                good += 1

            if (
                bi
                % GRAD_ACCUM_STEPS
                == 0
                or bi
                == len(refit_loader)
            ):
                if good:
                    refit_scaler.unscale_(
                        refit_optimizer
                    )
                    torch.nn.utils.clip_grad_norm_(
                        [
                            p
                            for p
                            in refit_model.parameters()
                            if p.requires_grad
                        ],
                        GRAD_CLIP,
                    )

                    old_scale = float(
                        refit_scaler.get_scale()
                    )
                    refit_scaler.step(
                        refit_optimizer
                    )
                    refit_scaler.update()
                    refit_optimizer.zero_grad(
                        set_to_none=True
                    )

                    if (
                        float(
                            refit_scaler.get_scale()
                        )
                        >= old_scale
                    ):
                        refit_scheduler.step()
                        refit_global_update += 1
                        refit_ema.update()
                else:
                    refit_optimizer.zero_grad(
                        set_to_none=True
                    )
                good = 0

            if (
                bi % 500 == 0
                or bi
                == len(refit_loader)
            ):
                elapsed = (
                    time.time() - t0
                ) / 60
                eta = (
                    elapsed
                    * (
                        len(refit_loader)
                        - bi
                    )
                    / max(1, bi)
                )
                print(
                    f"REFIT epoch="
                    f"{epoch}/"
                    f"{REFIT_EPOCHS} "
                    f"batch={bi}/"
                    f"{len(refit_loader)} "
                    f"loss="
                    f"{running/max(1,bi):.4f} "
                    f"elapsed="
                    f"{elapsed:.1f}m "
                    f"ETA={eta:.1f}m",
                    flush=True,
                )

    with refit_ema.applied():
        rids, rprob, _, ratt = (
            predict_loader(
                refit_model,
                test_loader,
            )
        )

    assert list(
        map(str, rids)
    ) == list(map(str, tids))

    np.savez_compressed(
        ROOT
        / "test_probs_refit_exp4.npz",
        id=np.asarray(rids),
        prob=rprob,
        thresholds=thresholds,
    )

    print(
        "✅ fresh TRAIN+DEV "
        "Prefix refit complete"
    )


Fresh refit epochs: 2
TRAIN+DEV rows: 52931
✅ refit prefix initialized from the same real-token K/V probe
refit prefix trainable: 147456
refit head trainable: 761573
REFIT epoch=1/2 batch=500/13233 loss=0.5453 elapsed=0.8m ETA=19.9m
REFIT epoch=1/2 batch=1000/13233 loss=0.4994 elapsed=1.6m ETA=19.1m
REFIT epoch=1/2 batch=1500/13233 loss=0.4782 elapsed=2.3m ETA=18.3m
REFIT epoch=1/2 batch=2000/13233 loss=0.4692 elapsed=3.1m ETA=17.5m
REFIT epoch=1/2 batch=2500/13233 loss=0.4619 elapsed=3.9m ETA=16.7m
REFIT epoch=1/2 batch=3000/13233 loss=0.4547 elapsed=4.7m ETA=15.9m
REFIT epoch=1/2 batch=3500/13233 loss=0.4481 elapsed=5.4m ETA=15.1m
REFIT epoch=1/2 batch=4000/13233 loss=0.4422 elapsed=6.2m ETA=14.4m
REFIT epoch=1/2 batch=4500/13233 loss=0.4371 elapsed=7.0m ETA=13.6m
REFIT epoch=1/2 batch=5000/13233 loss=0.4336 elapsed=7.8m ETA=12.8m
REFIT epoch=1/2 batch=5500/13233 loss=0.4304 elapsed=8.6m ETA=12.0m
REFIT epoch=1/2 batch=6000/13233 loss=0.4270 elapsed=9.3m ETA=11.3m
REFIT epoch=1/2 bat

In [19]:
# CELL 17 — REFIT + HELDOUT/REFIT ENSEMBLE ZIPs
def validate_submission_zip(
    zip_path,
    expected_ids,
):
    tmp = ROOT / (
        "check_" + zip_path.stem
    )
    if tmp.exists():
        shutil.rmtree(tmp)
    tmp.mkdir(
        parents=True,
        exist_ok=True,
    )

    with zipfile.ZipFile(
        zip_path, "r"
    ) as z:
        assert z.namelist() == [
            "prediction.json"
        ]
        z.extract(
            "prediction.json",
            tmp,
        )

    parsed = eval_exact.read_pred(
        str(
            tmp / "prediction.json"
        ),
        N_LABELS,
    )
    assert set(parsed) == expected_ids
    return parsed


expected_ids = {
    str(r["id"])
    for r in test_records
}

if not RUN_REFIT:
    print(
        "RUN_REFIT=False -> "
        "refit/ensemble ZIPs skipped."
    )
else:
    refit_records = (
        make_pred_records(
            rids,
            rprob,
            thresholds,
            True,
        )
    )

    REFIT_PRED = (
        ROOT
        / "prediction_refit.json"
    )
    json.dump(
        refit_records,
        open(
            REFIT_PRED,
            "w",
            encoding="utf-8",
        ),
        ensure_ascii=False,
    )

    REFIT_ZIP = (
        ROOT
        / "submission_exp4_Model1_Prefix_REFIT.zip"
    )
    with zipfile.ZipFile(
        REFIT_ZIP,
        "w",
        compression=
            zipfile.ZIP_DEFLATED,
    ) as z:
        z.write(
            REFIT_PRED,
            arcname="prediction.json",
        )

    validate_submission_zip(
        REFIT_ZIP,
        expected_ids,
    )

    ensemble_prob = (
        (
            1.0
            - REFIT_ENSEMBLE_WEIGHT
        )
        * tprob
        + REFIT_ENSEMBLE_WEIGHT
        * rprob
    )

    ensemble_records = (
        make_pred_records(
            tids,
            ensemble_prob,
            thresholds,
            True,
        )
    )

    ENSEMBLE_PRED = (
        ROOT
        / "prediction_heldout_refit_ensemble.json"
    )
    json.dump(
        ensemble_records,
        open(
            ENSEMBLE_PRED,
            "w",
            encoding="utf-8",
        ),
        ensure_ascii=False,
    )

    ENSEMBLE_ZIP = (
        ROOT
        / "submission_exp4_Model1_Prefix_HELDOUT_REFIT_ENSEMBLE.zip"
    )
    with zipfile.ZipFile(
        ENSEMBLE_ZIP,
        "w",
        compression=
            zipfile.ZIP_DEFLATED,
    ) as z:
        z.write(
            ENSEMBLE_PRED,
            arcname="prediction.json",
        )

    validate_submission_zip(
        ENSEMBLE_ZIP,
        expected_ids,
    )

    np.savez_compressed(
        ROOT
        / "test_probs_heldout_refit_ensemble_exp4.npz",
        id=np.asarray(tids),
        prob=ensemble_prob,
        thresholds=thresholds,
        refit_weight=
            REFIT_ENSEMBLE_WEIGHT,
    )

    print(
        "✅ REFIT ZIP PASS:",
        REFIT_ZIP,
    )
    print(
        "✅ HELDOUT+REFIT "
        "ENSEMBLE ZIP PASS:",
        ENSEMBLE_ZIP,
    )
    print(
        "\nCORE SUBMISSIONS:"
    )
    print(
        "1. HELDOUT :", ZIP
    )
    print(
        "2. REFIT   :", REFIT_ZIP
    )
    print(
        "3. ENSEMBLE:",
        ENSEMBLE_ZIP,
    )


✅ REFIT ZIP PASS: /kaggle/working/phase5_exp4_model1_prefix_fast_v3/submission_exp4_Model1_Prefix_REFIT.zip
✅ HELDOUT+REFIT ENSEMBLE ZIP PASS: /kaggle/working/phase5_exp4_model1_prefix_fast_v3/submission_exp4_Model1_Prefix_HELDOUT_REFIT_ENSEMBLE.zip

CORE SUBMISSIONS:
1. HELDOUT : /kaggle/working/phase5_exp4_model1_prefix_fast_v3/submission_exp4_Model1_Prefix_HELDOUT.zip
2. REFIT   : /kaggle/working/phase5_exp4_model1_prefix_fast_v3/submission_exp4_Model1_Prefix_REFIT.zip
3. ENSEMBLE: /kaggle/working/phase5_exp4_model1_prefix_fast_v3/submission_exp4_Model1_Prefix_HELDOUT_REFIT_ENSEMBLE.zip


In [20]:
# CELL 18 — experiment manifest
manifest = {
    "phase": 5,
    "experiment": 4,
    "version": NOTEBOOK_VERSION,
    "task":
        "Legal Statute Identification",
    "method":
        "Prefix Tuning",
    "phase1_model":
        "Model1 Encoder/Prefix-LM",
    "architecture": {
        "vocab_size": VOCAB_SIZE,
        "d_model": D_MODEL,
        "n_layers": N_LAYERS,
        "n_heads": N_HEADS,
        "ffn_dim": FFN_DIM,
        "max_positions":
            MAX_POSITIONS,
        "sinusoidal_positions":
            True,
        "pre_layernorm":
            True,
        "activation":
            "GELU",
    },
    "prefix": {
        "type":
            "continuous per-layer KV prefix",
        "prefix_len":
            PREFIX_LEN,
        "prefix_start_block":
            PREFIX_START_BLOCK,
        "prefix_blocks":
            PREFIX_BLOCKS,
        "prefix_only_params":
            prefix_trainable,
        "backbone_trainable":
            0,
        "initialization":
            "real-token K/V warm-start" if PREFIX_INIT_FROM_REAL_TOKENS else "Gaussian",
        "init_probe_view":
            "statute_evidence",
    },
    "speed_design": {
        "n_chunks": N_CHUNKS,
        "chunk_len": CHUNK_LEN,
        "views": VIEW_NAMES,
        "evals_per_epoch": 1,
        "epochs": EPOCHS,
        "frozen_prefix_blocks":
            PREFIX_START_BLOCK,
        "refit": RUN_REFIT,
        "refit_epochs":
            globals().get(
                "REFIT_EPOCHS"
            ),
    },
    "training": {
        "loss":
            "BCEWithLogits + sqrt(raw pos_weight) clipped [1,10]",
        "prefix_lr": PREFIX_LR,
        "head_lr": HEAD_LR,
        "micro_batch":
            MICRO_BATCH_SIZE,
        "grad_accum":
            GRAD_ACCUM_STEPS,
        "ema_decay": EMA_DECAY,
    },
    "selection": {
        "primary":
            "cross-fitted clean-DEV Macro-F1",
        "best_epoch":
            best["epoch"],
        "cv_macro_f1":
            best["cv_macro_f1"],
        "threshold_strategy":
            best[
                "threshold_strategy"
            ],
        "clean_metrics":
            best["clean_metrics"],
        "official_metrics":
            best[
                "official_metrics"
            ],
    },
    "submission_variants": {
        "heldout_zip":
            str(ZIP),
        "refit_zip":
            str(
                globals().get(
                    "REFIT_ZIP"
                )
            ),
        "heldout_refit_ensemble":
            str(
                globals().get(
                    "ENSEMBLE_ZIP"
                )
            ),
    },
    "assets": {
        "model1_sha256":
            sha256_file(
                MODEL1_CKPT
            ),
        "tokenizer_sha256":
            sha256_file(
                TOKENIZER_FILE
            ),
        "dataset_sha256":
            DATA_SHA256,
    },
    "reference": {
        "paper":
            "Li & Liang (2021), Prefix-Tuning: Optimizing Continuous Prompts for Generation",
        "doi":
            "10.18653/v1/2021.acl-long.353",
    },
}

json.dump(
    manifest,
    open(
        ROOT / "manifest.json",
        "w",
    ),
    indent=2,
)

print(
    json.dumps(
        manifest,
        indent=2,
    )
)


{
  "phase": 5,
  "experiment": 4,
  "version": "exp4_model1_prefix_fast_v3_realinit_refit_ensemble",
  "task": "Legal Statute Identification",
  "method": "Prefix Tuning",
  "phase1_model": "Model1 Encoder/Prefix-LM",
  "architecture": {
    "vocab_size": 24576,
    "d_model": 768,
    "n_layers": 14,
    "n_heads": 12,
    "ffn_dim": 3072,
    "max_positions": 2048,
    "sinusoidal_positions": true,
    "pre_layernorm": true,
    "activation": "GELU"
  },
  "prefix": {
    "type": "continuous per-layer KV prefix",
    "prefix_len": 16,
    "prefix_start_block": 8,
    "prefix_blocks": [
      8,
      9,
      10,
      11,
      12,
      13
    ],
    "prefix_only_params": 147456,
    "backbone_trainable": 0,
    "initialization": "real-token K/V warm-start",
    "init_probe_view": "statute_evidence"
  },
  "speed_design": {
    "n_chunks": 3,
    "chunk_len": 320,
    "views": [
      "head",
      "statute_evidence",
      "tail"
    ],
    "evals_per_epoch": 1,
    "epochs": 2,


## What to record after the run

For the report/viva:

- Model1 total parameters
- prefix-only trainable parameters
- total trainable percentage
- heldout epoch time
- best clean DEV Macro-F1
- official DEV Macro/Micro/Subset metrics
- threshold strategy
- refit time
- heldout / refit / ensemble Codabench scores
- comparison against Partial FT, LoRMA+ and AdaLoRA

The experiment counts as **Prefix Tuning** because all original Model1 weights remain frozen and task-specific continuous K/V prefixes are learned inside Transformer attention.


In [21]:
# CELL 19 (OPTIONAL) — blend Exp3 + Exp4 probabilities without retraining
# Upload Exp3's:
#   dev_probs_exp3.npz
#   test_probs_heldout_refit_ensemble_exp3.npz
# If absent, this cell safely skips.

EXP3_DEV_NPZ = None
EXP3_TEST_NPZ = None

def _find_named_npz(names):
    hits = []
    for base in [
        "/kaggle/input",
        "/kaggle/working",
    ]:
        if not os.path.exists(base):
            continue
        for root, _, files in os.walk(
            base
        ):
            for name in names:
                if name in files:
                    p = os.path.join(
                        root, name
                    )
                    if (
                        "phase5_exp4_model1_prefix"
                        not in p.lower()
                    ):
                        hits.append(p)
    return (
        sorted(set(hits))[0]
        if hits else None
    )

EXP3_DEV_NPZ = (
    EXP3_DEV_NPZ
    or _find_named_npz(
        ["dev_probs_exp3.npz"]
    )
)
EXP3_TEST_NPZ = (
    EXP3_TEST_NPZ
    or _find_named_npz(
        [
            "test_probs_heldout_refit_ensemble_exp3.npz",
            "test_probs_heldout_exp3.npz",
        ]
    )
)

print("Exp3 DEV :", EXP3_DEV_NPZ)
print("Exp3 TEST:", EXP3_TEST_NPZ)

if not (
    EXP3_DEV_NPZ
    and EXP3_TEST_NPZ
):
    print(
        "⚠️ Exp3 probability files "
        "not found -> blend skipped."
    )
else:
    d3 = np.load(
        EXP3_DEV_NPZ,
        allow_pickle=True,
    )
    t3 = np.load(
        EXP3_TEST_NPZ,
        allow_pickle=True,
    )

    def _align(
        ids_src,
        prob_src,
        ids_dst,
        label,
    ):
        pos = {
            str(i): k
            for k, i
            in enumerate(ids_src)
        }
        missing = [
            str(i)
            for i in ids_dst
            if str(i) not in pos
        ]
        assert not missing, (
            label,
            len(missing),
            missing[:5],
        )
        return prob_src[
            [
                pos[str(i)]
                for i in ids_dst
            ]
        ]

    p3_dev = _align(
        d3["id"],
        d3["prob"],
        dids,
        "Exp3 DEV",
    )
    p3_test = _align(
        t3["id"],
        t3["prob"],
        tids,
        "Exp3 TEST",
    )

    cy = dy[CLEAN_IDX]
    rows = []

    for w in np.round(
        np.arange(
            0.0, 1.01, 0.1
        ),
        2,
    ):
        # w = Exp4 heldout weight
        pb = (
            w * dprob
            + (1.0-w) * p3_dev
        )
        tune_w, _, _ = (
            optimize_thresholds(
                cy,
                pb[CLEAN_IDX],
            )
        )
        rows.append(
            (
                float(w),
                tune_w[
                    "cv_macro_f1"
                ],
                tune_w["name"],
            )
        )
        print(
            f"w_exp4={w:.1f} "
            f"CV="
            f"{tune_w['cv_macro_f1']:.4f} "
            f"{tune_w['name']}"
        )

    w_best, best_cv, _ = max(
        rows,
        key=lambda r: r[1],
    )

    pb_dev = (
        w_best * dprob
        + (1.0-w_best) * p3_dev
    )
    tune_b, _, _ = (
        optimize_thresholds(
            cy,
            pb_dev[CLEAN_IDX],
        )
    )
    thr_b = np.asarray(
        tune_b["thresholds"],
        dtype=np.float32,
    )

    exp4_test_prob = (
        ensemble_prob
        if RUN_REFIT
        else tprob
    )
    pb_test = (
        w_best * exp4_test_prob
        + (1.0-w_best) * p3_test
    )

    blend_records = (
        make_pred_records(
            tids,
            pb_test,
            thr_b,
            True,
        )
    )

    BLEND_PRED = (
        ROOT
        / "prediction_exp3_exp4_blend.json"
    )
    json.dump(
        blend_records,
        open(
            BLEND_PRED,
            "w",
            encoding="utf-8",
        ),
        ensure_ascii=False,
    )

    BLEND_ZIP = (
        ROOT
        / "submission_exp3_exp4_BLEND.zip"
    )
    with zipfile.ZipFile(
        BLEND_ZIP,
        "w",
        compression=
            zipfile.ZIP_DEFLATED,
    ) as z:
        z.write(
            BLEND_PRED,
            arcname=
                "prediction.json",
        )

    validate_submission_zip(
        BLEND_ZIP,
        expected_ids,
    )
    print(
        "✅ EXP3+EXP4 BLEND ZIP PASS:",
        BLEND_ZIP,
    )


Exp3 DEV : None
Exp3 TEST: None
⚠️ Exp3 probability files not found -> blend skipped.
